# 04 - Pseudo-Label Generation from Grad-CAM

Extract pseudo-label segmentation masks từ Grad-CAM heatmaps.
- Toàn bộ ảnh tập train (tất cả các lớp)
- Morphological post-processing (CLOSE + OPEN)
- Visualization: Ảnh gốc | GradCAM overlay | Pseudo mask overlay (mỗi lớp 1 ảnh)

In [ ]:
import os
import sys

# Auto-detect project root
notebook_dir = os.path.abspath('')
proj_root = os.path.dirname(notebook_dir) if os.path.basename(notebook_dir) == 'notebooks' else notebook_dir
sys.path.insert(0, proj_root)
sys.path.insert(0, os.path.join(proj_root, 'utils'))

import torch
import numpy as np
import cv2
import json
import glob
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
from PIL import Image
from collections import defaultdict
from torchvision import transforms

from utils.models import EfficientNetClassifier
from utils.gradcam import GradCAMGenerator, find_efficientnet_target_layer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1) Load data + model

In [ ]:
# Đường dẫn tập train
train_dir = os.path.join(proj_root, 'notebooks', 'data', 'raw', 'train')
if not os.path.exists(train_dir):
    train_dir = 'data/raw/train'

print('Train dir:', train_dir)

# Lấy danh sách class
class_dirs = sorted([d for d in Path(train_dir).iterdir() if d.is_dir()])
class_names = [d.name for d in class_dirs]
class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = {i: name for name, i in class_to_idx.items()}
num_classes = len(class_names)

print(f'Classes ({num_classes}):', class_names)

# Lấy toàn bộ ảnh mỗi lớp (không giới hạn)
selected_samples = []

for cls_dir in class_dirs:
    cls_name = cls_dir.name
    cls_idx = class_to_idx[cls_name]
    imgs = sorted(list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')))
    for img_path in imgs:
        selected_samples.append((str(img_path), cls_idx, cls_name))
    print(f'  {cls_name}: {len(imgs)} ảnh')

print(f'Tổng: {len(selected_samples)} ảnh')

In [ ]:
# Load model
model = EfficientNetClassifier(num_classes=num_classes, backbone='efficientnet_b0', pretrained=False)

ckpt_patterns = [
    os.path.join(proj_root, 'models', 'classification', 'checkpoints', '*.pth'),
    'models/classification/checkpoints/*.pth',
]
candidates = []
for pat in ckpt_patterns:
    candidates.extend(glob.glob(pat))

if not candidates:
    raise FileNotFoundError('Không tìm thấy checkpoint. Chạy notebook 02 trước.')

ckpt_path = max(candidates, key=os.path.getctime)
checkpoint = torch.load(ckpt_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device).eval()
print('Loaded:', ckpt_path)

## 2) Grad-CAM setup

In [ ]:
target_layer = find_efficientnet_target_layer(model)
gradcam_gen = GradCAMGenerator(model, target_layer=target_layer, device=device.type)
print('Grad-CAM target layer:', target_layer)

infer_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 3) Pseudo-label extraction — toàn bộ ảnh train

In [ ]:
# Output dirs
pseudo_dir = Path(os.path.join(proj_root, 'notebooks', 'data', 'pseudo_labels_train'))
pseudo_dir.mkdir(parents=True, exist_ok=True)

# Cũng lưu ảnh gốc vào processed_train để dùng cho segmentation
processed_dir = Path(os.path.join(proj_root, 'notebooks', 'data', 'processed_train'))
processed_dir.mkdir(parents=True, exist_ok=True)

stats = []
results_by_class = defaultdict(list)

for img_path, cls_idx, cls_name in tqdm(selected_samples, desc='Pseudo-label generation'):
    try:
        # Load ảnh gốc
        orig_img = Image.open(img_path).convert('RGB').resize((224, 224))
        orig_arr = np.array(orig_img)

        # Tensor
        input_tensor = infer_transform(orig_img).unsqueeze(0).to(device)

        # Grad-CAM
        heatmap, confidence, pred_class = gradcam_gen.generate(input_tensor, cls_idx)

        # Tạo pseudo mask
        mask = (heatmap >= 0.5).astype(np.uint8)
        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        # Tên file
        fname = Path(img_path).name  # e.g. to_label_100.jpg
        base_name = f'{cls_name}_{fname}'  # thêm class prefix để tránh trùng

        # Lưu mask
        mask_save_path = pseudo_dir / f'{base_name}_pseudo.png'
        cv2.imwrite(str(mask_save_path), (mask * 255).astype(np.uint8))

        # Lưu ảnh gốc (resize 224x224)
        img_save_path = processed_dir / base_name
        orig_img.save(str(img_save_path))

        # GradCAM overlay
        heatmap_colored = cv2.applyColorMap((heatmap * 255).astype(np.uint8), cv2.COLORMAP_JET)
        heatmap_rgb = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
        gradcam_overlay = cv2.addWeighted(orig_arr, 0.6, heatmap_rgb, 0.4, 0)

        # Pseudo mask overlay
        mask_rgb = np.zeros_like(orig_arr)
        mask_rgb[:, :, 1] = mask * 255  # green channel
        pseudo_overlay = cv2.addWeighted(orig_arr, 0.7, mask_rgb, 0.3, 0)

        stat = {
            'filename': base_name,
            'original_path': img_path,
            'class_name': cls_name,
            'class_idx': cls_idx,
            'pred_class': pred_class,
            'confidence': float(confidence),
            'mask_coverage': float(mask.mean()),
        }
        stats.append(stat)

        results_by_class[cls_name].append({
            'orig_arr': orig_arr,
            'heatmap': heatmap,
            'gradcam_overlay': gradcam_overlay,
            'mask': mask,
            'pseudo_overlay': pseudo_overlay,
            'confidence': float(confidence),
            'pred_class': pred_class,
            'true_class': cls_idx,
        })

    except Exception as e:
        print(f'  Lỗi {img_path}: {e}')

# Lưu stats
with open(pseudo_dir / 'pseudo_label_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(f'\nGenerated pseudo-labels: {len(stats)}')
print(f'Saved to: {pseudo_dir}')

## 4) Visualization - Mỗi lớp 1 ảnh: Gốc | GradCAM overlay | Pseudo overlay

In [ ]:
# Visualization: mỗi lớp 1 ảnh đại diện (best confidence)
# Cột: Ảnh gốc | GradCAM overlay | Pseudo mask overlay

n_classes = len(class_names)
fig, axes = plt.subplots(n_classes, 3, figsize=(15, 5 * n_classes))
if n_classes == 1:
    axes = axes.reshape(1, -1)

for i, cls_name in enumerate(class_names):
    items = results_by_class.get(cls_name, [])
    if not items:
        continue
    # Chọn ảnh có confidence cao nhất
    best = max(items, key=lambda x: x['confidence'])

    axes[i, 0].imshow(best['orig_arr'])
    axes[i, 0].set_title(f'{cls_name}\nẢnh gốc', fontsize=10)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(best['gradcam_overlay'])
    axes[i, 1].set_title(f'GradCAM Overlay\nConf={best["confidence"]:.3f}', fontsize=10)
    axes[i, 1].axis('off')

    axes[i, 2].imshow(best['pseudo_overlay'])
    axes[i, 2].set_title(f'Pseudo Mask Overlay\nCoverage={best["mask"].mean():.3f}', fontsize=10)
    axes[i, 2].axis('off')

plt.suptitle('Pseudo-Label Visualization (Best sample per class)', fontsize=14)
plt.tight_layout()

out_dir = os.path.join(proj_root, 'models', 'classification', 'gradcam')
os.makedirs(out_dir, exist_ok=True)
save_path = os.path.join(out_dir, 'pseudo_label_visualization.png')
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

In [ ]:
# Visualization thêm: So sánh trước/sau morphological operations
# Chọn 1 ảnh bất kỳ để demo
demo_cls = class_names[0]
demo_items = results_by_class.get(demo_cls, [])

if demo_items:
    demo = demo_items[0]
    orig_arr = demo['orig_arr']
    heatmap = demo['heatmap']

    # Mask trước morphological
    mask_raw = (heatmap >= 0.5).astype(np.uint8)
    mask_raw_overlay = orig_arr.copy()
    mask_raw_rgb = np.zeros_like(orig_arr)
    mask_raw_rgb[:, :, 0] = mask_raw * 255  # red
    mask_raw_overlay = cv2.addWeighted(orig_arr, 0.7, mask_raw_rgb, 0.3, 0)

    # Mask sau morphological (đã lưu trong demo['mask'])
    mask_morph = demo['mask']
    mask_morph_rgb = np.zeros_like(orig_arr)
    mask_morph_rgb[:, :, 1] = mask_morph * 255  # green
    mask_morph_overlay = cv2.addWeighted(orig_arr, 0.7, mask_morph_rgb, 0.3, 0)

    fig2, axes2 = plt.subplots(1, 4, figsize=(20, 5))
    axes2[0].imshow(orig_arr)
    axes2[0].set_title(f'Ảnh gốc\n{demo_cls}', fontsize=10)
    axes2[0].axis('off')

    axes2[1].imshow(heatmap, cmap='jet')
    axes2[1].set_title('GradCAM Heatmap', fontsize=10)
    axes2[1].axis('off')

    axes2[2].imshow(mask_raw_overlay)
    axes2[2].set_title(f'Mask trước Morphological\n(threshold=0.5)', fontsize=10)
    axes2[2].axis('off')

    axes2[3].imshow(mask_morph_overlay)
    axes2[3].set_title(f'Mask sau Morphological\n(CLOSE+OPEN 5x5)', fontsize=10)
    axes2[3].axis('off')

    plt.suptitle('Morphological Processing Demo', fontsize=13)
    plt.tight_layout()
    save_path2 = os.path.join(out_dir, 'morphological_demo.png')
    plt.savefig(save_path2, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path2}')

## 5) Đánh giá chất lượng pseudo-label

In [ ]:
import pandas as pd

df = pd.DataFrame(stats)
print('=== Pseudo-label Statistics ===')
print(df[['class_name', 'confidence', 'mask_coverage']].groupby('class_name').describe())

print(f'\nTổng số pseudo-labels: {len(df)}')
print(f'Accuracy (pred==true): {(df["pred_class"] == df["class_idx"]).mean():.2%}')
print(f'Low coverage (<0.1): {(df["mask_coverage"] < 0.1).sum()}')
print(f'Good coverage (>=0.1): {(df["mask_coverage"] >= 0.1).sum()}')